OWASP ZAP Scanned Output

In [14]:
from bs4 import BeautifulSoup
import pandas as pd
from pathlib import Path
import re
from urllib.parse import urlparse

# =============================
# CONFIGURATION
# =============================
TARGET_APP = "DVWA"
TOOL_NAME = "OWASP_ZAP"
SCAN_RUN_PREFIX = f"{TARGET_APP}_ZAP"

HTML_FILES = [
    Path("2026-05-30-ZAP-Report-Medium_1.htm"),
]

if not 1 <= len(HTML_FILES) <= 5:
    raise ValueError("HTML_FILES must contain between 1 and 5 files.")

# =============================
# HELPERS
# =============================

def build_scan_run_id(html_path: Path, index: int) -> str:
    stem = re.sub(r"[^A-Za-z0-9]+", "_", html_path.stem).strip("_")
    return f"{SCAN_RUN_PREFIX}_{index:02d}_{stem}"


def clean_text(text: str) -> str:
    text = (text or "").replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def map_zap_risk_to_severity(risk_text: str) -> str:
    risk = (risk_text or "").strip()
    if risk.lower() == "info":
        return "Informational"
    return risk


def map_to_owasp_2025_from_cwe(cwe_id: str) -> str:
    """
    Conservative mapping:
    - use only the CWE already stated by ZAP in the report
    - do not infer CWE from keywords
    - map only from explicit CWE values found in the report
    """
    cwe = str(cwe_id or "").replace("CWE-", "").strip()

    if not cwe:
        return ""

    mapping = {
        "79": "A03",    # Cross-Site Scripting
        "89": "A03",    # SQL Injection
        "352": "A01",   # CSRF
        "538": "A01",   # File and Directory Information Exposure
        "548": "A01",   # Information Exposure Through Directory Listing
        "550": "A01",   # Information Exposure Through Error Message
        "552": "A01",   # Files or Directories Accessible to External Parties
        "565": "A01",   # Reliance on Cookies without Validation / cookie poisoning style weakness
        "598": "A02",   # Sensitive info in query strings
        "693": "A05",   # Protection Mechanism Failure
        "1004": "A02",  # Cookie without HttpOnly
        "1104": "A06",  # Use of unmaintained / vulnerable components
    }

    return mapping.get(cwe, "")


def extract_elapsed_time_minutes(soup: BeautifulSoup) -> float:
    text = soup.get_text("\n", strip=True)

    patterns = [
        r'Elapsed Time[:\s]+(\d+(?:\.\d+)?)\s*seconds',
        r'Duration[:\s]+(\d{1,2}):(\d{2}):(\d{2})',
        r'Time taken[:\s]+(\d+(?:\.\d+)?)\s*seconds'
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            if len(match.groups()) == 1:
                return round(float(match.group(1)) / 60.0, 2)
            elif len(match.groups()) == 3:
                h, m, s = map(int, match.groups())
                return round((h * 3600 + m * 60 + s) / 60.0, 2)

    return 0.0


def infer_target_fields_from_url(url: str):
    result = {
        "Target_IP": "",
        "Target_Hostname": "",
        "Target_Port": "",
        "Node_Name_Default": "",
    }

    if not url:
        return result

    parsed = urlparse(url)
    result["Target_Hostname"] = parsed.hostname or ""
    result["Target_Port"] = str(parsed.port) if parsed.port else ""
    result["Node_Name_Default"] = parsed.path or url

    host = parsed.hostname or ""
    if re.fullmatch(r"(?:\d{1,3}\.){3}\d{1,3}", host):
        result["Target_IP"] = host

    return result


def normalize_parameter(value: str) -> str:
    value = clean_text(value)
    return value if value else "N/A"


def normalize_references(value: str) -> str:
    return clean_text(value)


def is_alert_table(table):
    rows = table.find_all("tr")
    if len(rows) < 2:
        return False

    first_row = [clean_text(cell.get_text(" ", strip=True)) for cell in rows[0].find_all(["td", "th"])]
    if len(first_row) != 2:
        return False

    valid_risks = {"High", "Medium", "Low", "Informational", "Info"}
    return first_row[0] in valid_risks


def parse_alert_table(table, scan_run_id: str):
    rows = table.find_all("tr")
    parsed_rows = [
        [clean_text(cell.get_text(" ", strip=True)) for cell in tr.find_all(["td", "th"])]
        for tr in rows
    ]
    parsed_rows = [r for r in parsed_rows if len(r) > 0]

    if len(parsed_rows) < 2:
        return []

    first = parsed_rows[0]
    if len(first) < 2:
        return []

    risk_level = first[0].strip()
    vuln_name = first[1].strip()

    description = ""
    if len(parsed_rows) > 1 and len(parsed_rows[1]) >= 2 and parsed_rows[1][0] == "Description":
        description = parsed_rows[1][1]

    global_fields = {
        "Instances": "",
        "Solution": "",
        "Reference": "",
        "CWE Id": "",
        "WASC Id": "",
        "Plugin Id": ""
    }

    findings = []
    current_instance = None

    def flush_instance():
        nonlocal current_instance
        if current_instance is not None:
            findings.append(current_instance.copy())

    for row in parsed_rows[2:]:
        if len(row) == 1 and row[0] == "":
            continue

        if len(row) >= 2:
            key = row[0].strip()
            value = row[1].strip()

            if key == "URL":
                flush_instance()
                current_instance = {
                    "URL": value,
                    "Node Name": "",
                    "Method": "",
                    "Parameter": "",
                    "Attack": "",
                    "Evidence": "",
                    "Other Info": ""
                }
            elif key in ["Node Name", "Method", "Parameter", "Attack", "Evidence", "Other Info"]:
                if current_instance is not None:
                    current_instance[key] = value
            elif key in global_fields:
                global_fields[key] = value

    flush_instance()

    output = []

    for inst in findings:
        cwe_numeric = clean_text(global_fields["CWE Id"])
        cwe_id = f"CWE-{cwe_numeric}" if cwe_numeric else ""
        owasp_cat = map_to_owasp_2025_from_cwe(cwe_numeric)

        url = inst.get("URL", "")
        parameter = normalize_parameter(inst.get("Parameter", ""))
        method = clean_text(inst.get("Method", ""))
        attack = clean_text(inst.get("Attack", ""))
        evidence = clean_text(inst.get("Evidence", ""))
        other_info = clean_text(inst.get("Other Info", ""))
        node_name = clean_text(inst.get("Node Name", ""))

        inferred_target = infer_target_fields_from_url(url)
        if not node_name:
            node_name = inferred_target["Node_Name_Default"]

        canonical_key = f"{TARGET_APP}:{url}:{parameter}:{vuln_name}"

        output.append({
            "Target_App": TARGET_APP,
            "Tool_Name": TOOL_NAME,
            "Scan_Run_ID": scan_run_id,
            "Vuln_ID_Tool": clean_text(global_fields["Plugin Id"]),
            "HTTP_Method": method,
            "URL_or_Endpoint": url,
            "Parameter": parameter,
            "Vuln_Type_Label": vuln_name,
            "Description": description,
            "CWE_ID": cwe_id,
            "OWASP_2025_Category": owasp_cat,
            "Severity_Level": map_zap_risk_to_severity(risk_level),
            "Confidence_Level": "",   # not provided in the ZAP DVWA report
            "First_Detected_Timestamp": "",
            "Scan_Duration_Min": None,
            "Is_Duplicate_Within_Tool": "No",
            "Canonical_Vuln_Key": canonical_key,
            "References": normalize_references(global_fields["Reference"]),
            "Target_IP": inferred_target["Target_IP"],
            "Target_Hostname": inferred_target["Target_Hostname"],
            "Target_Port": inferred_target["Target_Port"],
            "Node_Name": node_name,
            "Attack": attack,
            "Evidence": evidence,
            "Other_Info": other_info,
            "Solution": clean_text(global_fields["Solution"]),
            "Instances_Reported_By_ZAP": clean_text(global_fields["Instances"]),
            "WASC_ID": clean_text(global_fields["WASC Id"]),
            "Plugin_ID": clean_text(global_fields["Plugin Id"]),
            "Source_HTML_File": ""
        })

    return output


# =============================
# MAIN PARSER
# =============================

def parse_zap_html_report(html_path: Path, scan_run_id: str):
    if not html_path.exists():
        raise FileNotFoundError(f"File not found: {html_path}")

    html = html_path.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(html, "html.parser")

    elapsed_min = extract_elapsed_time_minutes(soup)

    all_rows = []

    for table in soup.find_all("table"):
        if is_alert_table(table):
            parsed = parse_alert_table(table, scan_run_id)
            all_rows.extend(parsed)

    df = pd.DataFrame(all_rows)

    if not df.empty:
        df["Scan_Duration_Min"] = elapsed_min
        df["Source_HTML_File"] = html_path.name

        df = df.drop_duplicates(
            subset=[
                "Scan_Run_ID",
                "URL_or_Endpoint",
                "HTTP_Method",
                "Parameter",
                "Vuln_Type_Label",
                "Plugin_ID",
                "Attack",
                "Evidence"
            ]
        ).reset_index(drop=True)

    return df, elapsed_min


# =============================
# RUN
# =============================

all_dfs = []
scan_run_map = []

for idx, html_file in enumerate(HTML_FILES, start=1):
    scan_run_id = build_scan_run_id(html_file, idx)
    df, elapsed_min = parse_zap_html_report(html_file, scan_run_id)

    all_dfs.append(df)
    scan_run_map.append({
        "HTML_FILE": str(html_file),
        "SCAN_RUN_ID": scan_run_id,
        "Elapsed_Minutes": elapsed_min,
        "Findings_Count": len(df)
    })

    print(f"Processed {html_file} -> {scan_run_id} | findings={len(df)} | elapsed={elapsed_min} min")

zap_df = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()
scan_runs_df = pd.DataFrame(scan_run_map)

print(f"\nProcessed {len(HTML_FILES)} file(s).")
print(f"Total findings parsed: {len(zap_df)}")

zap_df.to_csv("zap_per_vulnerability_html_multi_dvwa.csv", index=False)
print("Saved per-vulnerability dataset to zap_per_vulnerability_html_multi_dvwa.csv")

scan_runs_df.to_csv("zap_scan_run_mapping_dvwa.csv", index=False)
print("Saved scan-run mapping to zap_scan_run_mapping_dvwa.csv")

display(scan_runs_df)

if not zap_df.empty:
    display(
        zap_df[
            [
                "Vuln_ID_Tool",
                "HTTP_Method",
                "URL_or_Endpoint",
                "Parameter",
                "Vuln_Type_Label",
                "CWE_ID",
                "OWASP_2025_Category",
                "Severity_Level",
                "Instances_Reported_By_ZAP"
            ]
        ].head(20)
    )

# =============================
# OVERLAP / UNIQUENESS TABLE
# =============================

if not zap_df.empty:
    overlap_rows = []

    unique_keys = zap_df[["Scan_Run_ID", "Canonical_Vuln_Key"]].drop_duplicates()

    for _, row in unique_keys.iterrows():
        subset = zap_df[
            (zap_df["Scan_Run_ID"] == row["Scan_Run_ID"]) &
            (zap_df["Canonical_Vuln_Key"] == row["Canonical_Vuln_Key"])
        ]
        first = subset.iloc[0]

        overlap_rows.append({
            "Scan_Run_ID": first["Scan_Run_ID"],
            "Target_App": first["Target_App"],
            "Canonical_Vuln_Key": first["Canonical_Vuln_Key"],
            "URL_or_Endpoint": first["URL_or_Endpoint"],
            "Parameter": first["Parameter"],
            "Vuln_Type_Label": first["Vuln_Type_Label"],
            "CWE_ID": first["CWE_ID"],
            "OWASP_2025_Category": first["OWASP_2025_Category"],
            "Detected_by_BurpSuite": "No",
            "Detected_by_OWASP_ZAP": "Yes",
            "Detected_by_Nikto": "No",
            "Overlap_Pattern": "OWASP_ZAP_only"
        })

    overlap_df = pd.DataFrame(overlap_rows)
    overlap_df.to_csv("overlap_table_zap_seed_html_multi_dvwa.csv", index=False)
    print("Saved overlap table seed to overlap_table_zap_seed_html_multi_dvwa.csv")

    display(overlap_df.head())
else:
    print("No findings parsed, so overlap table was not created.")

Processed 2026-05-30-ZAP-Report-Medium_1.htm -> DVWA_ZAP_01_2026_05_30_ZAP_Report_Medium_1 | findings=131 | elapsed=0.0 min

Processed 1 file(s).
Total findings parsed: 131
Saved per-vulnerability dataset to zap_per_vulnerability_html_multi_dvwa.csv
Saved scan-run mapping to zap_scan_run_mapping_dvwa.csv


,HTML_FILE,SCAN_RUN_ID,Elapsed_Minutes,Findings_Count
0,2026-05-30-ZAP-Report-Medium_1.htm,DVWA_ZAP_01_2026_05_30_ZAP_Report_Medium_1,0.0,131


,Vuln_ID_Tool,HTTP_Method,URL_or_Endpoint,Parameter,Vuln_Type_Label,CWE_ID,OWASP_2025_Category,Severity_Level,Instances_Reported_By_ZAP
0,40018,GET,http://127.0.0.1/DVWA/vulnerabilities/xss_d/?d...,default,SQL Injection,CWE-89,A03,High,7
1,40018,POST,http://127.0.0.1/DVWA/vulnerabilities/captcha/,Change,SQL Injection,CWE-89,A03,High,7
2,40018,POST,http://127.0.0.1/DVWA/vulnerabilities/captcha/...,Change,SQL Injection,CWE-89,A03,High,7
3,40018,POST,http://127.0.0.1/DVWA/vulnerabilities/csp/,N/A,SQL Injection,CWE-89,A03,High,7
4,40018,POST,http://127.0.0.1/DVWA/vulnerabilities/upload/,Upload,SQL Injection,CWE-89,A03,High,7
5,40018,POST,http://127.0.0.1/DVWA/vulnerabilities/xss_s/,btnSign,SQL Injection,CWE-89,A03,High,7
6,40018,POST,http://127.0.0.1/DVWA/vulnerabilities/xss_s/,txtName,SQL Injection,CWE-89,A03,High,7
7,10202,GET,http://127.0.0.1/DVWA/vulnerabilities/csp/,N/A,Absence of Anti-CSRF Tokens,CWE-352,A01,Medium,4
8,10202,GET,http://127.0.0.1/DVWA/vulnerabilities/weak_id/,N/A,Absence of Anti-CSRF Tokens,CWE-352,A01,Medium,4
9,10202,POST,http://127.0.0.1/DVWA/vulnerabilities/csp/,N/A,Absence of Anti-CSRF Tokens,CWE-352,A01,Medium,4


Saved overlap table seed to overlap_table_zap_seed_html_multi_dvwa.csv


,Scan_Run_ID,Target_App,Canonical_Vuln_Key,URL_or_Endpoint,Parameter,Vuln_Type_Label,CWE_ID,OWASP_2025_Category,Detected_by_BurpSuite,Detected_by_OWASP_ZAP,Detected_by_Nikto,Overlap_Pattern
0,DVWA_ZAP_01_2026_05_30_ZAP_Report_Medium_1,DVWA,DVWA:http://127.0.0.1/DVWA/vulnerabilities/xss...,http://127.0.0.1/DVWA/vulnerabilities/xss_d/?d...,default,SQL Injection,CWE-89,A03,No,Yes,No,OWASP_ZAP_only
1,DVWA_ZAP_01_2026_05_30_ZAP_Report_Medium_1,DVWA,DVWA:http://127.0.0.1/DVWA/vulnerabilities/cap...,http://127.0.0.1/DVWA/vulnerabilities/captcha/,Change,SQL Injection,CWE-89,A03,No,Yes,No,OWASP_ZAP_only
2,DVWA_ZAP_01_2026_05_30_ZAP_Report_Medium_1,DVWA,DVWA:http://127.0.0.1/DVWA/vulnerabilities/cap...,http://127.0.0.1/DVWA/vulnerabilities/captcha/...,Change,SQL Injection,CWE-89,A03,No,Yes,No,OWASP_ZAP_only
3,DVWA_ZAP_01_2026_05_30_ZAP_Report_Medium_1,DVWA,DVWA:http://127.0.0.1/DVWA/vulnerabilities/csp...,http://127.0.0.1/DVWA/vulnerabilities/csp/,N/A,SQL Injection,CWE-89,A03,No,Yes,No,OWASP_ZAP_only
4,DVWA_ZAP_01_2026_05_30_ZAP_Report_Medium_1,DVWA,DVWA:http://127.0.0.1/DVWA/vulnerabilities/upl...,http://127.0.0.1/DVWA/vulnerabilities/upload/,Upload,SQL Injection,CWE-89,A03,No,Yes,No,OWASP_ZAP_only


Nikto Scanned Output

In [12]:
from bs4 import BeautifulSoup
import pandas as pd
from pathlib import Path
import re
from urllib.parse import urlparse

# =============================
# CONFIGURATION
# =============================
TARGET_APP = "DVWA"
TOOL_NAME = "Nikto"
SCAN_RUN_PREFIX = f"{TARGET_APP}_NIKTO"

HTML_FILES = [
    Path("20260530_dvwa_medium_2.html.htm"),
]

if not 1 <= len(HTML_FILES) <= 5:
    raise ValueError("HTML_FILES must contain between 1 and 5 files.")

# =============================
# HELPERS
# =============================

def build_scan_run_id(html_path: Path, index: int) -> str:
    stem = re.sub(r"[^A-Za-z0-9]+", "_", html_path.stem).strip("_")
    return f"{SCAN_RUN_PREFIX}_{index:02d}_{stem}"


def clean_text(text: str) -> str:
    text = (text or "").replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def extract_reference_ids(ref_text: str):
    ref_text = clean_text(ref_text)

    cve_matches = re.findall(r"\bCVE-\d{4}-\d+\b", ref_text, flags=re.I)
    cwe_matches = re.findall(r"\bCWE-\d+\b", ref_text, flags=re.I)
    osvdb_matches = re.findall(r"\bOSVDB[-: ]\d+\b", ref_text, flags=re.I)

    normalized = []
    for x in cve_matches:
        normalized.append(x.upper())
    for x in cwe_matches:
        normalized.append(x.upper())
    for x in osvdb_matches:
        normalized.append(x.upper().replace(":", "-").replace(" ", "-"))

    return list(dict.fromkeys(normalized))


def extract_explicit_cwe(description: str, references: str) -> str:
    combined = f"{description} {references}"
    m = re.search(r"\bCWE-\d+\b", combined, flags=re.I)
    return m.group(0).upper() if m else ""


def build_vuln_type_label(description: str, uri: str) -> str:
    d = clean_text(description)

    patterns = [
        (r"Directory indexing found", "Directory indexing"),
        (r"Configuration information may be available remotely", "Configuration information exposed"),
        (r"Database directory found", "Database directory exposed"),
        (r"Admin login page/section found", "Admin login page exposed"),
        (r"Git Index file may contain directory listing information", "Git index exposed"),
        (r"Git HEAD file found", "Git HEAD exposed"),
        (r"Git config file found", "Git config exposed"),
        (r"\.gitignore file found", ".gitignore file exposed"),
        (r"\.dockerignore file found", ".dockerignore file exposed"),
        (r"Suggested security header missing:\s*([A-Za-z0-9\-_]+)", None),
        (r"This might be interesting", "Interesting file or directory"),
        (r"X-Frame-Options header is deprecated", "Deprecated security header usage"),
        (r"The X-Content-Type-Options header is not set", "Missing X-Content-Type-Options header"),
    ]

    for pattern, fixed_label in patterns:
        m = re.search(pattern, d, flags=re.I)
        if m:
            if fixed_label is None:
                header_name = m.group(1)
                return f"Missing security header: {header_name}"
            return fixed_label

    prefix_removed = d
    if uri and d.startswith(uri):
        prefix_removed = d[len(uri):].lstrip(" :.-")

    if "." in prefix_removed:
        prefix_removed = prefix_removed.split(".")[0].strip()

    return prefix_removed[:120] if prefix_removed else "Nikto finding"


def map_cwe_to_owasp_2025(cwe_id: str) -> str:
    cwe = cwe_id.replace("CWE-", "").strip()

    if cwe in {"79", "89"}:
        return "A03"
    if cwe in {"1004"}:
        return "A02"
    if cwe in {"552", "548", "200"}:
        return "A01"
    if cwe in {"1104"}:
        return "A06"
    return ""


def map_to_cwe_and_owasp(description: str, references: str, uri: str, link: str):
    d = clean_text(description)
    d_lower = d.lower()
    refs = clean_text(references)
    refs_lower = refs.lower()

    explicit_cwe = extract_explicit_cwe(d, refs)
    if explicit_cwe:
        return explicit_cwe, map_cwe_to_owasp_2025(explicit_cwe)

    if "directory indexing found" in d_lower:
        cwe = "CWE-548"
        return cwe, map_cwe_to_owasp_2025(cwe)

    if (
        "configuration information may be available remotely" in d_lower
        or "git head file found" in d_lower
        or "git config file found" in d_lower
        or "git index file may contain directory listing information" in d_lower
    ):
        cwe = "CWE-200"
        return cwe, map_cwe_to_owasp_2025(cwe)

    if ".gitignore file found" in d_lower or ".dockerignore file found" in d_lower:
        cwe = "CWE-200"
        return cwe, map_cwe_to_owasp_2025(cwe)

    if "x-content-type-options header is not set" in d_lower:
        return "", ""

    if "suggested security header missing" in d_lower:
        return "", ""

    if "x-frame-options header is deprecated" in d_lower:
        return "", ""

    if "database directory found" in d_lower:
        return "", ""

    if "admin login page/section found" in d_lower:
        return "", ""

    if "this might be interesting" in d_lower:
        return "", ""

    if "appears to be outdated" in d_lower and ("cve-" in refs_lower or "osvdb" in refs_lower):
        cwe = "CWE-1104"
        return cwe, map_cwe_to_owasp_2025(cwe)

    return "", ""


def get_top_metadata(soup: BeautifulSoup):
    meta = {
        "Target_IP": "",
        "Target_Hostname": "",
        "Target_Port": "",
        "HTTP_Server": "",
        "Site_Link_Name": "",
        "Site_Link_IP": "",
        "Elapsed_Time_Sec": 0.0,
    }

    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) < 2:
                continue

            label = clean_text(tds[0].get_text(" ", strip=True))
            value = clean_text(tds[1].get_text(" ", strip=True))

            if label == "Target IP":
                meta["Target_IP"] = value
            elif label == "Target hostname":
                meta["Target_Hostname"] = value
            elif label == "Target Port":
                meta["Target_Port"] = value
            elif label == "HTTP Server":
                meta["HTTP_Server"] = value
            elif label == "Site Link (Name)":
                meta["Site_Link_Name"] = value
            elif label == "Site Link (IP)":
                meta["Site_Link_IP"] = value
            elif label == "Elapsed Time":
                m = re.search(r"(\d+(?:\.\d+)?)", value)
                if m:
                    meta["Elapsed_Time_Sec"] = float(m.group(1))

    return meta


def parse_finding_table(table):
    finding = {
        "URI": "",
        "HTTP_Method": "",
        "Description": "",
        "Link": "",
        "References": "",
    }

    rows = table.find_all("tr")
    for tr in rows:
        cells = tr.find_all("td")
        if len(cells) != 2:
            continue

        label = clean_text(cells[0].get_text(" ", strip=True))
        value = clean_text(cells[1].get_text(" ", strip=True))

        if label == "URI":
            finding["URI"] = value
        elif label == "HTTP Method":
            finding["HTTP_Method"] = value
        elif label == "Description":
            finding["Description"] = value
        elif label == "Link":
            finding["Link"] = value
        elif label.startswith("Reference"):
            finding["References"] = value

    return finding


def is_finding_table(table):
    rows = table.find_all("tr")
    if len(rows) < 3:
        return False

    labels = []
    for tr in rows[:5]:
        cells = tr.find_all("td")
        if len(cells) >= 1:
            labels.append(clean_text(cells[0].get_text(" ", strip=True)))

    return "URI" in labels and "HTTP Method" in labels and "Description" in labels


def parse_nikto_html_findings(html_path: Path, scan_run_id: str):
    if not html_path.exists():
        raise FileNotFoundError(f"File not found: {html_path}")

    with html_path.open("r", encoding="utf-8", errors="ignore") as f:
        soup = BeautifulSoup(f, "html.parser")

    meta = get_top_metadata(soup)
    elapsed_min = round(meta["Elapsed_Time_Sec"] / 60.0, 2) if meta["Elapsed_Time_Sec"] else 0.0

    rows = []
    finding_counter = 0

    for table in soup.find_all("table"):
        if not is_finding_table(table):
            continue

        finding = parse_finding_table(table)
        uri = finding["URI"]
        http_method = finding["HTTP_Method"]
        description = finding["Description"]
        link = finding["Link"]
        references = finding["References"]

        if not uri and not description:
            continue

        finding_counter += 1
        vuln_id_tool = f"NIKTO-{finding_counter:03d}"

        vuln_type_label = build_vuln_type_label(description, uri)
        cwe_id, owasp_cat = map_to_cwe_and_owasp(description, references, uri, link)

        parsed_link = urlparse(link) if link else None
        url_or_endpoint = link if link else uri
        node_name = parsed_link.path if parsed_link and parsed_link.path else uri

        canonical_key = f"{TARGET_APP}:{url_or_endpoint}:N/A:{vuln_type_label}"

        reference_ids = extract_reference_ids(references)

        other_info_parts = []
        if meta["HTTP_Server"]:
            other_info_parts.append(f"HTTP Server: {meta['HTTP_Server']}")
        if references:
            other_info_parts.append(f"Reference(s): {references}")

        row = {
            "Target_App": TARGET_APP,
            "Tool_Name": TOOL_NAME,
            "Scan_Run_ID": scan_run_id,
            "Vuln_ID_Tool": vuln_id_tool,
            "HTTP_Method": http_method,
            "URL_or_Endpoint": url_or_endpoint,
            "Parameter": "N/A",
            "Vuln_Type_Label": vuln_type_label,
            "Description": description,
            "CWE_ID": cwe_id,
            "OWASP_2025_Category": owasp_cat,
            "Severity_Level": "",
            "Confidence_Level": "",
            "First_Detected_Timestamp": "",
            "Scan_Duration_Min": elapsed_min,
            "Is_Duplicate_Within_Tool": "No",
            "Canonical_Vuln_Key": canonical_key,
            "References": references,
            "Target_IP": meta["Target_IP"],
            "Target_Hostname": meta["Target_Hostname"],
            "Target_Port": meta["Target_Port"],
            "Node_Name": node_name,
            "Attack": "",
            "Evidence": ", ".join(reference_ids),
            "Other_Info": " | ".join(other_info_parts),
            "Solution": "",
            "Instances_Reported_By_ZAP": "",
            "WASC_ID": "",
            "Plugin_ID": "",
            "Source_HTML_File": html_path.name,
        }

        rows.append(row)

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df.drop_duplicates(
            subset=[
                "Scan_Run_ID",
                "URL_or_Endpoint",
                "HTTP_Method",
                "Vuln_Type_Label",
                "Description"
            ]
        ).reset_index(drop=True)

    return df, elapsed_min


# =============================
# RUN
# =============================

all_dfs = []
scan_run_map = []

for idx, html_file in enumerate(HTML_FILES, start=1):
    scan_run_id = build_scan_run_id(html_file, idx)
    df, elapsed_min = parse_nikto_html_findings(html_file, scan_run_id)

    all_dfs.append(df)
    scan_run_map.append({
        "HTML_FILE": str(html_file),
        "SCAN_RUN_ID": scan_run_id,
        "Elapsed_Minutes": elapsed_min,
        "Findings_Count": len(df)
    })

    print(f"Processed {html_file} -> {scan_run_id} | findings={len(df)} | elapsed={elapsed_min} min")

nikto_df = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()
scan_runs_df = pd.DataFrame(scan_run_map)

print(f"\nProcessed {len(HTML_FILES)} file(s).")
print(f"Total findings parsed: {len(nikto_df)}")

nikto_df.to_csv("nikto_per_vulnerability_html_multi_dvwa.csv", index=False)
print("Saved per-vulnerability dataset to nikto_per_vulnerability_html_multi_dvwa.csv")

scan_runs_df.to_csv("nikto_scan_run_mapping_dvwa.csv", index=False)
print("Saved scan-run mapping to nikto_scan_run_mapping_dvwa.csv")

display(scan_runs_df)

if not nikto_df.empty:
    display(
        nikto_df[
            [
                "Vuln_ID_Tool",
                "HTTP_Method",
                "URL_or_Endpoint",
                "Vuln_Type_Label",
                "CWE_ID",
                "OWASP_2025_Category",
                "Severity_Level",
                "Confidence_Level"
            ]
        ].head(20)
    )

# =============================
# OVERLAP / UNIQUENESS TABLE
# =============================

if not nikto_df.empty:
    unique_keys = nikto_df[["Scan_Run_ID", "Canonical_Vuln_Key"]].drop_duplicates()
    overlap_rows = []

    for _, row in unique_keys.iterrows():
        subset = nikto_df[
            (nikto_df["Scan_Run_ID"] == row["Scan_Run_ID"]) &
            (nikto_df["Canonical_Vuln_Key"] == row["Canonical_Vuln_Key"])
        ]
        first = subset.iloc[0]

        overlap_rows.append({
            "Scan_Run_ID": first["Scan_Run_ID"],
            "Target_App": first["Target_App"],
            "Canonical_Vuln_Key": first["Canonical_Vuln_Key"],
            "URL_or_Endpoint": first["URL_or_Endpoint"],
            "Parameter": first["Parameter"],
            "Vuln_Type_Label": first["Vuln_Type_Label"],
            "CWE_ID": first["CWE_ID"],
            "OWASP_2025_Category": first["OWASP_2025_Category"],
            "Detected_by_BurpSuite": "No",
            "Detected_by_OWASP_ZAP": "No",
            "Detected_by_Nikto": "Yes",
            "Overlap_Pattern": "Nikto_only",
        })

    overlap_df = pd.DataFrame(overlap_rows)
    overlap_df.to_csv("overlap_table_nikto_seed_html_multi_dvwa.csv", index=False)
    print("Saved overlap table seed to overlap_table_nikto_seed_html_multi_dvwa.csv")

    display(overlap_df.head())
else:
    print("No findings parsed; overlap table not created.")

Processed 20260530_dvwa_medium_2.html.htm -> DVWA_NIKTO_01_20260530_dvwa_medium_2_html | findings=20 | elapsed=5.52 min

Processed 1 file(s).
Total findings parsed: 20
Saved per-vulnerability dataset to nikto_per_vulnerability_html_multi_dvwa.csv
Saved scan-run mapping to nikto_scan_run_mapping_dvwa.csv


,HTML_FILE,SCAN_RUN_ID,Elapsed_Minutes,Findings_Count
0,20260530_dvwa_medium_2.html.htm,DVWA_NIKTO_01_20260530_dvwa_medium_2_html,5.52,20


,Vuln_ID_Tool,HTTP_Method,URL_or_Endpoint,Vuln_Type_Label,CWE_ID,OWASP_2025_Category,Severity_Level,Confidence_Level
0,NIKTO-001,GET,http://127.0.0.1:80/DVWA/,Missing security header: x-content-type-options,,,,
1,NIKTO-002,GET,http://127.0.0.1:80/DVWA/,Missing security header: content-security-policy,,,,
2,NIKTO-003,GET,http://127.0.0.1:80/DVWA/,Missing security header: strict-transport-secu...,,,,
3,NIKTO-004,GET,http://127.0.0.1:80/DVWA/,Missing security header: permissions-policy,,,,
4,NIKTO-005,GET,http://127.0.0.1:80/DVWA/,Missing security header: referrer-policy,,,,
5,NIKTO-006,GET,http://127.0.0.1:80/DVWA/config/,Directory indexing,CWE-548,A01,,
6,NIKTO-007,GET,http://127.0.0.1:80/DVWA/config/,Configuration information exposed,CWE-200,A01,,
7,NIKTO-008,GET,http://127.0.0.1:80/DVWA/tests/,Directory indexing,CWE-548,A01,,
8,NIKTO-009,GET,http://127.0.0.1:80/DVWA/tests/,Interesting file or directory,,,,
9,NIKTO-010,GET,http://127.0.0.1:80/DVWA/database/,Directory indexing,CWE-548,A01,,


Saved overlap table seed to overlap_table_nikto_seed_html_multi_dvwa.csv


,Scan_Run_ID,Target_App,Canonical_Vuln_Key,URL_or_Endpoint,Parameter,Vuln_Type_Label,CWE_ID,OWASP_2025_Category,Detected_by_BurpSuite,Detected_by_OWASP_ZAP,Detected_by_Nikto,Overlap_Pattern
0,DVWA_NIKTO_01_20260530_dvwa_medium_2_html,DVWA,DVWA:http://127.0.0.1:80/DVWA/:N/A:Missing sec...,http://127.0.0.1:80/DVWA/,N/A,Missing security header: x-content-type-options,,,No,No,Yes,Nikto_only
1,DVWA_NIKTO_01_20260530_dvwa_medium_2_html,DVWA,DVWA:http://127.0.0.1:80/DVWA/:N/A:Missing sec...,http://127.0.0.1:80/DVWA/,N/A,Missing security header: content-security-policy,,,No,No,Yes,Nikto_only
2,DVWA_NIKTO_01_20260530_dvwa_medium_2_html,DVWA,DVWA:http://127.0.0.1:80/DVWA/:N/A:Missing sec...,http://127.0.0.1:80/DVWA/,N/A,Missing security header: strict-transport-secu...,,,No,No,Yes,Nikto_only
3,DVWA_NIKTO_01_20260530_dvwa_medium_2_html,DVWA,DVWA:http://127.0.0.1:80/DVWA/:N/A:Missing sec...,http://127.0.0.1:80/DVWA/,N/A,Missing security header: permissions-policy,,,No,No,Yes,Nikto_only
4,DVWA_NIKTO_01_20260530_dvwa_medium_2_html,DVWA,DVWA:http://127.0.0.1:80/DVWA/:N/A:Missing sec...,http://127.0.0.1:80/DVWA/,N/A,Missing security header: referrer-policy,,,No,No,Yes,Nikto_only


Burp Suite Scanned Output

In [2]:
from bs4 import BeautifulSoup, Tag
import pandas as pd
from pathlib import Path
import re

# =============================
# CONFIGURATION
# =============================
TARGET_APP = "DVWA"
TOOL_NAME = "BurpSuite_Professional"
SCAN_RUN_PREFIX = f"{TARGET_APP}_BURP"

HTML_FILES = [
    Path("2026-05-25-BURP-DVWA-Report-4.html"),
]

if not 1 <= len(HTML_FILES) <= 5:
    raise ValueError("HTML_FILES must contain between 1 and 5 files.")

# =============================
# HELPERS
# =============================

def build_scan_run_id(html_path: Path, index: int) -> str:
    stem = re.sub(r"[^A-Za-z0-9]+", "_", html_path.stem).strip("_")
    return f"{SCAN_RUN_PREFIX}_{index:02d}_{stem}"


def clean_text(text: str) -> str:
    text = (text or "").replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def map_burp_severity(sev_text: str) -> str:
    sev = clean_text(sev_text).lower()
    if sev == "high":
        return "High"
    if sev == "medium":
        return "Medium"
    if sev == "low":
        return "Low"
    if sev in ("information", "informational", "info"):
        return "Informational"
    if sev == "false positive":
        return "False Positive"
    return "Unknown"


def map_burp_confidence(conf_text: str) -> str:
    conf = clean_text(conf_text).lower()
    if conf == "certain":
        return "Certain"
    if conf == "firm":
        return "Firm"
    if conf == "tentative":
        return "Tentative"
    return "Unknown"


def infer_cwe_from_vuln_name(vuln_name: str) -> str:
    name = (vuln_name or "").lower()

    if "cross-site scripting" in name or "cross site scripting" in name or "xss" in name:
        return "CWE-79"
    if "sql injection" in name:
        return "CWE-89"
    if "php code injection" in name or "code injection" in name:
        return "CWE-94"
    if "xml injection" in name:
        return "CWE-91"
    if "path traversal" in name or "file path manipulation" in name:
        return "CWE-22"
    if "open redirection" in name or "open redirect" in name or "long redirection response" in name:
        return "CWE-601"
    if "csrf" in name or "cross-site request forgery" in name:
        return "CWE-352"
    if "cookie without httponly flag set" in name:
        return "CWE-1004"
    if "cleartext submission of password" in name or "unencrypted communications" in name:
        return "CWE-319"
    if "password submitted using get method" in name:
        return "CWE-598"
    if "client-side desync" in name:
        return "CWE-444"
    return ""


# def map_to_owasp_2025(vuln_name: str, cwe_id: str = "") -> str:
#     name = (vuln_name or "").lower()
#     cwe = str(cwe_id).replace("CWE-", "").strip()

#     if cwe in ["79", "89", "77", "78", "90", "91", "93", "94", "98", "643", "917", "444"]:
#         return "A03"
#     if any(k in name for k in [
#         "cross-site scripting", "cross site scripting", "xss",
#         "sql injection", "command injection", "ldap injection",
#         "xpath injection", "code injection", "php code injection",
#         "xml injection", "client-side desync"
#     ]):
#         return "A03"

#     if cwe in ["22", "23", "36", "59", "200", "201", "284", "285", "352", "425",
#                "538", "548", "601", "639", "862", "863", "918"]:
#         return "A01"
#     if any(k in name for k in [
#         "path-relative style sheet import",
#         "cross-site request forgery",
#         "csrf",
#         "long redirection response",
#         "backup file"
#     ]):
#         return "A01"
    
#     if cwe in ["16", "611", "614", "942", "1004", "1104"]:
#         return "A02"
#     if any(k in name for k in [
#         "cookie without httponly flag set",
#         "frameable response",
#         "cross-domain referer leakage",
#         "content security policy"
#     ]):
#         return "A02"

#     if cwe in ["261", "319", "326", "327", "328", "330", "331", "523", "759", "760", "916"]:
#         return "A02"
#     if any(k in name for k in [
#         "cleartext submission of password",
#         "unencrypted communications"
#     ]):
#         return "A02"

#     return ""

def map_to_owasp_2025(vuln_name: str, cwe_id: str = "") -> str:
    name = (vuln_name or "").lower().strip()
    cwe = str(cwe_id or "").replace("CWE-", "").strip()

    # A05:2025 - Injection
    # Official 2025 list places Injection at A05.
    if cwe in ["79", "89", "77", "78", "90", "91", "93", "94", "98", "643", "917"]:
        return "A05"
    if any(k in name for k in [
        "cross-site scripting",
        "cross site scripting",
        "xss",
        "sql injection",
        "command injection",
        "ldap injection",
        "xpath injection",
        "code injection",
        "php code injection",
        "xml injection"
    ]):
        return "A05"

    # A03:2025 - Software Supply Chain Failures
    # 2025 expands vulnerable/outdated components into supply-chain failures.
    if cwe in ["1104", "829"]:
        return "A03"
    if any(k in name for k in [
        "vulnerable javascript dependency",
        "vulnerable js dependency",
        "outdated javascript library",
        "outdated js library",
        "outdated component",
        "vulnerable component",
        "cross-domain script include"
    ]):
        return "A03"

    # A04:2025 - Cryptographic Failures
    if cwe in ["261", "319", "326", "327", "328", "330", "331", "523", "759", "760", "916"]:
        return "A04"
    if any(k in name for k in [
        "cleartext submission of password",
        "unencrypted communications",
        "weak ssl",
        "weak tls",
        "cryptographic"
    ]):
        return "A04"

    # A01:2025 - Broken Access Control
    if cwe in [
        "22", "23", "36", "59", "61", "65",
        "200", "201", "219", "276", "281", "282", "283",
        "284", "285", "352", "359", "377", "379",
        "402", "424", "425", "441", "497", "538", "540",
        "548", "552", "566", "601", "615", "639",
        "668", "732", "749", "862", "863", "918", "922", "1275"
    ]:
        return "A01"
    if any(k in name for k in [
        "cross-site request forgery",
        "csrf",
        "open redirect",
        "open redirection",
        "directory browsing",
        "directory indexing",
        "source code disclosure",
        "git index exposed",
        "git head exposed",
        "git config exposed",
        ".gitignore file exposed",
        ".dockerignore file exposed",
        "session token in url"
    ]):
        return "A01"

    # A02:2025 - Security Misconfiguration
    if cwe in ["16", "444", "611", "614", "942", "1004"]:
        return "A02"
    if any(k in name for k in [
        "client-side desync",
        "frameable response",
        "frameable response (potential clickjacking)",
        "path-relative style sheet import",
        "cookie without httponly flag set",
        "cross-domain referer leakage",
        "backup file"
    ]):
        return "A02"

    # Intentionally left unmapped:
    # - long redirection response
    # - private ip addresses disclosed
    # - robots.txt file
    # These are often informational / weakly aligned and should not be forced
    # into an OWASP Top 10:2025 category without stronger justification.

    return ""


def extract_elapsed_time_minutes(raw_html: str) -> float:
    text = raw_html
    patterns = [
        r'Elapsed Time[:\s]+(\d+(?:\.\d+)?)\s*seconds',
        r'Duration[:\s]+(\d{1,2}):(\d{2}):(\d{2})',
        r'Time taken[:\s]+(\d+(?:\.\d+)?)\s*seconds'
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            if len(match.groups()) == 1:
                return round(float(match.group(1)) / 60.0, 2)
            if len(match.groups()) == 3:
                h, m, s = map(int, match.groups())
                return round((h * 3600 + m * 60 + s) / 60.0, 2)
    return 0.0


def parse_summary_table(section_html: str):
    out = {
        "Severity_Level": "Unknown",
        "Confidence_Level": "Unknown",
        "Target_Hostname": "",
        "Node_Name": "",
        "URL_or_Endpoint": ""
    }

    summary_match = re.search(
        r'<h2>\s*Summary\s*</h2>(.*?)(?:<h2>|$)',
        section_html,
        flags=re.I | re.S
    )
    if not summary_match:
        return out

    summary_html = summary_match.group(1)
    summary_soup = BeautifulSoup(summary_html, "html.parser")
    table = summary_soup.find("table", class_="summarytable")
    if not table:
        return out

    rows = table.find_all("tr")
    kv = {}

    for tr in rows:
        cells = tr.find_all(["td", "th"])
        texts = [clean_text(cell.get_text(" ", strip=True)).rstrip(":") for cell in cells]
        texts = [t for t in texts if t]
        if len(texts) >= 2:
            for i in range(len(texts) - 1):
                key = texts[i].lower()
                val = texts[i + 1]
                if key in ("severity", "confidence", "host", "path") and key not in kv:
                    kv[key] = val

    sev = kv.get("severity", "")
    conf = kv.get("confidence", "")
    host = kv.get("host", "")
    path = kv.get("path", "")

    out["Severity_Level"] = map_burp_severity(sev)
    out["Confidence_Level"] = map_burp_confidence(conf)
    out["Target_Hostname"] = host
    out["Node_Name"] = path

    if host and path:
        if path.startswith("/"):
            out["URL_or_Endpoint"] = host.rstrip("/") + path
        else:
            out["URL_or_Endpoint"] = host.rstrip("/") + "/" + path

    return out


def extract_text_after_heading(section_html: str, heading: str) -> str:
    pattern = rf'<h2>\s*{re.escape(heading)}\s*</h2>(.*?)(?:<h2>|$)'
    m = re.search(pattern, section_html, flags=re.I | re.S)
    if not m:
        return ""
    chunk = m.group(1)
    soup = BeautifulSoup(chunk, "html.parser")
    return clean_text(soup.get_text(" ", strip=True))


def extract_cwe_from_classifications(classifications_text: str) -> str:
    m = re.search(r'\bCWE-\d+\b', classifications_text, flags=re.I)
    return m.group(0).upper() if m else ""


def extract_parameter_from_issue_number_text(issue_number_text: str) -> str:
    m = re.search(r'\[([^\]]+)\]', issue_number_text)
    if not m:
        return "N/A"

    raw = clean_text(m.group(1))
    raw = re.sub(r'\s+parameter$', '', raw, flags=re.I).strip()
    return raw if raw else "N/A"


def extract_url_from_issue_number_text(issue_number_text: str) -> str:
    m = re.search(r'https?://[^\s\[]+', issue_number_text, flags=re.I)
    return m.group(0).strip() if m else ""


def get_toc_vulnerabilities(soup: BeautifulSoup):
    issues = []
    current_severity_group = ""

    for p in soup.find_all("p", class_=re.compile(r"^TOCH[01]$")):
        a = p.find("a", href=True)
        if not a:
            continue

        href = a.get("href", "").strip()
        text = clean_text(a.get_text(" ", strip=True))
        p_class = " ".join(p.get("class", []))

        if "TOCH0" in p_class:
            current_severity_group = re.sub(r'^\d+\.\s*', '', text).strip()

        elif "TOCH1" in p_class:
            issue_number_match = re.match(r'^(\d+)\.\s*(.+)$', text)
            if issue_number_match:
                issue_num = issue_number_match.group(1)
                vuln_name = issue_number_match.group(2).strip()
            else:
                issue_num = href.lstrip("#")
                vuln_name = re.sub(r'^\d+\.\s*', '', text).strip()

            issues.append({
                "anchor": href.lstrip("#"),
                "issue_number": issue_num,
                "vuln_name": vuln_name,
                "severity_group": current_severity_group
            })

    return issues


def get_issue_section_html(raw_html: str, anchor_id: str, next_anchor_id: str | None) -> str:
    start_pat = rf'<span[^>]*id="{re.escape(anchor_id)}"[^>]*>'
    start = re.search(start_pat, raw_html, flags=re.I)
    if not start:
        return ""

    start_idx = start.start()

    if next_anchor_id:
        end_pat = rf'<span[^>]*id="{re.escape(next_anchor_id)}"[^>]*>'
        end = re.search(end_pat, raw_html[start_idx + 1:], flags=re.I)
        if end:
            end_idx = start_idx + 1 + end.start()
            return raw_html[start_idx:end_idx]

    return raw_html[start_idx:]


def extract_instance_headers(section_html: str):
    soup = BeautifulSoup(section_html, "html.parser")
    headers = []

    for p in soup.find_all("p", class_="TOCH1"):
        txt = clean_text(p.get_text(" ", strip=True))
        if re.match(r'^\d+\.\d+\.\s+https?://', txt, flags=re.I):
            headers.append(txt)

    if headers:
        return headers

    text = soup.get_text("\n", strip=True)
    regex_headers = re.findall(r'^\d+\.\d+\.\s+https?://.+$', text, flags=re.M)
    return [clean_text(x) for x in regex_headers]


def split_summary_tables(section_html: str):
    soup = BeautifulSoup(section_html, "html.parser")
    summary_tables = []

    for h2 in soup.find_all("h2"):
        if clean_text(h2.get_text(" ", strip=True)).lower() == "summary":
            table = h2.find_next("table", class_="summarytable")
            if table:
                summary_tables.append(str(table))

    return summary_tables


def parse_one_summary_table(table_html: str):
    out = {
        "Severity_Level": "Unknown",
        "Confidence_Level": "Unknown",
        "Target_Hostname": "",
        "Node_Name": "",
        "URL_or_Endpoint": ""
    }

    soup = BeautifulSoup(table_html, "html.parser")
    table = soup.find("table")
    if not table:
        return out

    rows = table.find_all("tr")
    kv = {}

    for tr in rows:
        cells = tr.find_all(["td", "th"])
        texts = [clean_text(cell.get_text(" ", strip=True)).rstrip(":") for cell in cells]
        texts = [t for t in texts if t]
        if len(texts) >= 2:
            for i in range(len(texts) - 1):
                key = texts[i].lower()
                val = texts[i + 1]
                if key in ("severity", "confidence", "host", "path") and key not in kv:
                    kv[key] = val

    sev = kv.get("severity", "")
    conf = kv.get("confidence", "")
    host = kv.get("host", "")
    path = kv.get("path", "")

    out["Severity_Level"] = map_burp_severity(sev)
    out["Confidence_Level"] = map_burp_confidence(conf)
    out["Target_Hostname"] = host
    out["Node_Name"] = path

    if host and path:
        if path.startswith("/"):
            out["URL_or_Endpoint"] = host.rstrip("/") + path
        else:
            out["URL_or_Endpoint"] = host.rstrip("/") + "/" + path

    return out


def parse_burp_html_report(html_path: Path, scan_run_id: str):
    if not html_path.exists():
        raise FileNotFoundError(f"File not found: {html_path}")

    raw_html = html_path.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(raw_html, "html.parser")
    elapsed_min = extract_elapsed_time_minutes(raw_html)

    toc_issues = get_toc_vulnerabilities(soup)
    rows = []

    for i, issue in enumerate(toc_issues):
        next_anchor = toc_issues[i + 1]["anchor"] if i + 1 < len(toc_issues) else None
        section_html = get_issue_section_html(raw_html, issue["anchor"], next_anchor)

        if not section_html:
            continue

        issue_detail = extract_text_after_heading(section_html, "Issue detail")
        issue_background = extract_text_after_heading(section_html, "Issue background")
        issue_remediation = (
            extract_text_after_heading(section_html, "Issue remediation") or
            extract_text_after_heading(section_html, "Remediation background") or
            extract_text_after_heading(section_html, "Remediation detail")
        )
        vuln_classifications = extract_text_after_heading(section_html, "Vulnerability classifications")
        references = extract_text_after_heading(section_html, "References")

        cwe_id = extract_cwe_from_classifications(vuln_classifications)
        if not cwe_id:
            cwe_id = infer_cwe_from_vuln_name(issue["vuln_name"])

        owasp_cat = map_to_owasp_2025(issue["vuln_name"], cwe_id)
        description_parts = [x for x in [issue_detail, issue_background] if x]
        description = " ".join(description_parts)

        instance_headers = extract_instance_headers(section_html)
        summary_tables = split_summary_tables(section_html)

        if summary_tables:
            for idx2, table_html in enumerate(summary_tables):
                summary = parse_one_summary_table(table_html)

                instance_text = instance_headers[idx2] if idx2 < len(instance_headers) else ""
                parameter = extract_parameter_from_issue_number_text(instance_text)
                url_from_header = extract_url_from_issue_number_text(instance_text)

                url_or_endpoint = summary["URL_or_Endpoint"] or url_from_header
                target_hostname = summary["Target_Hostname"]
                node_name = summary["Node_Name"]

                canonical_key = f"{TARGET_APP}:{url_or_endpoint}:{parameter}:{issue['vuln_name']}"

                rows.append({
                    "Target_App": TARGET_APP,
                    "Tool_Name": TOOL_NAME,
                    "Scan_Run_ID": scan_run_id,
                    "Vuln_ID_Tool": issue["issue_number"],
                    "HTTP_Method": "",
                    "URL_or_Endpoint": url_or_endpoint,
                    "Parameter": parameter,
                    "Vuln_Type_Label": issue["vuln_name"],
                    "Description": description,
                    "CWE_ID": cwe_id,
                    "OWASP_2025_Category": owasp_cat,
                    "Severity_Level": summary["Severity_Level"] if summary["Severity_Level"] != "Unknown" else map_burp_severity(issue["severity_group"]),
                    "Confidence_Level": summary["Confidence_Level"],
                    "First_Detected_Timestamp": "",
                    "Scan_Duration_Min": elapsed_min,
                    "Is_Duplicate_Within_Tool": "No",
                    "Canonical_Vuln_Key": canonical_key,
                    "References": references,
                    "Target_IP": "",
                    "Target_Hostname": target_hostname,
                    "Target_Port": "",
                    "Node_Name": node_name,
                    "Attack": "",
                    "Evidence": "",
                    "Other_Info": vuln_classifications,
                    "Solution": issue_remediation,
                    "Instances_Reported_By_ZAP": "",
                    "WASC_ID": "",
                    "Plugin_ID": "",
                    "Source_HTML_File": html_path.name
                })

        else:
            summary = parse_summary_table(section_html)
            canonical_key = f"{TARGET_APP}:{summary['URL_or_Endpoint']}:N/A:{issue['vuln_name']}"

            rows.append({
                "Target_App": TARGET_APP,
                "Tool_Name": TOOL_NAME,
                "Scan_Run_ID": scan_run_id,
                "Vuln_ID_Tool": issue["issue_number"],
                "HTTP_Method": "",
                "URL_or_Endpoint": summary["URL_or_Endpoint"],
                "Parameter": "N/A",
                "Vuln_Type_Label": issue["vuln_name"],
                "Description": description,
                "CWE_ID": cwe_id,
                "OWASP_2025_Category": owasp_cat,
                "Severity_Level": summary["Severity_Level"] if summary["Severity_Level"] != "Unknown" else map_burp_severity(issue["severity_group"]),
                "Confidence_Level": summary["Confidence_Level"],
                "First_Detected_Timestamp": "",
                "Scan_Duration_Min": elapsed_min,
                "Is_Duplicate_Within_Tool": "No",
                "Canonical_Vuln_Key": canonical_key,
                "References": references,
                "Target_IP": "",
                "Target_Hostname": summary["Target_Hostname"],
                "Target_Port": "",
                "Node_Name": summary["Node_Name"],
                "Attack": "",
                "Evidence": "",
                "Other_Info": vuln_classifications,
                "Solution": issue_remediation,
                "Instances_Reported_By_ZAP": "",
                "WASC_ID": "",
                "Plugin_ID": "",
                "Source_HTML_File": html_path.name
            })

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df.drop_duplicates(
            subset=[
                "Scan_Run_ID",
                "URL_or_Endpoint",
                "Parameter",
                "Vuln_Type_Label",
                "Severity_Level",
                "Confidence_Level"
            ]
        ).reset_index(drop=True)

    return df, elapsed_min


# =============================
# RUN
# =============================
all_dfs = []
scan_run_map = []

for idx, html_file in enumerate(HTML_FILES, start=1):
    scan_run_id = build_scan_run_id(html_file, idx)
    df, elapsed_min = parse_burp_html_report(html_file, scan_run_id)

    all_dfs.append(df)
    scan_run_map.append({
        "HTML_FILE": str(html_file),
        "SCAN_RUN_ID": scan_run_id,
        "Elapsed_Minutes": elapsed_min,
        "Findings_Count": len(df)
    })

    print(f"Processed {html_file} -> {scan_run_id} | findings={len(df)} | elapsed={elapsed_min} min")

burp_df = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()
scan_runs_df = pd.DataFrame(scan_run_map)

print(f"\nProcessed {len(HTML_FILES)} file(s).")
print(f"Total findings parsed: {len(burp_df)}")

burp_df.to_csv("burp_per_vulnerability_html_multi_dvwa.csv", index=False)
print("Saved per-vulnerability dataset to burp_per_vulnerability_html_multi_dvwa.csv")

scan_runs_df.to_csv("burp_scan_run_mapping_dvwa.csv", index=False)
print("Saved scan-run mapping to burp_scan_run_mapping_dvwa.csv")

display(scan_runs_df)

if not burp_df.empty:
    display(
        burp_df[
            [
                "Vuln_ID_Tool",
                "Vuln_Type_Label",
                "URL_or_Endpoint",
                "Parameter",
                "Severity_Level",
                "Confidence_Level",
                "CWE_ID",
                "OWASP_2025_Category"
            ]
        ].head(30)
    )

# =============================
# OVERLAP / UNIQUENESS TABLE
# =============================
if not burp_df.empty:
    unique_keys = burp_df[["Scan_Run_ID", "Canonical_Vuln_Key"]].drop_duplicates()
    overlap_rows = []

    for _, row in unique_keys.iterrows():
        subset = burp_df[
            (burp_df["Scan_Run_ID"] == row["Scan_Run_ID"]) &
            (burp_df["Canonical_Vuln_Key"] == row["Canonical_Vuln_Key"])
        ]
        first = subset.iloc[0]

        overlap_rows.append({
            "Scan_Run_ID": first["Scan_Run_ID"],
            "Target_App": first["Target_App"],
            "Canonical_Vuln_Key": first["Canonical_Vuln_Key"],
            "URL_or_Endpoint": first["URL_or_Endpoint"],
            "Parameter": first["Parameter"],
            "Vuln_Type_Label": first["Vuln_Type_Label"],
            "CWE_ID": first["CWE_ID"],
            "OWASP_2025_Category": first["OWASP_2025_Category"],
            "Detected_by_BurpSuite": "Yes",
            "Detected_by_OWASP_ZAP": "No",
            "Detected_by_Nikto": "No",
            "Overlap_Pattern": "BurpSuite_only",
        })

    overlap_df = pd.DataFrame(overlap_rows)
    overlap_df.to_csv("overlap_table_burp_seed_html_multi_dvwa.csv", index=False)
    print("Saved overlap table seed to overlap_table_burp_seed_html_multi_dvwa.csv")

    display(overlap_df.head())
else:
    print("No findings parsed; overlap table not created.")

Processed 2026-05-25-BURP-DVWA-Report-4.html -> DVWA_BURP_01_2026_05_25_BURP_DVWA_Report_4 | findings=31 | elapsed=0.0 min

Processed 1 file(s).
Total findings parsed: 31
Saved per-vulnerability dataset to burp_per_vulnerability_html_multi_dvwa.csv
Saved scan-run mapping to burp_scan_run_mapping_dvwa.csv


,HTML_FILE,SCAN_RUN_ID,Elapsed_Minutes,Findings_Count
0,2026-05-25-BURP-DVWA-Report-4.html,DVWA_BURP_01_2026_05_25_BURP_DVWA_Report_4,0.0,31


,Vuln_ID_Tool,Vuln_Type_Label,URL_or_Endpoint,Parameter,Severity_Level,Confidence_Level,CWE_ID,OWASP_2025_Category
0,1,1. Cross-site scripting (reflected),,N/A,Unknown,Unknown,CWE-79,A05
1,1,2. Cross-site scripting (DOM-based),,N/A,Unknown,Unknown,CWE-79,A05
2,1,3. Cleartext submission of password,,N/A,Unknown,Unknown,CWE-319,A04
3,2,1. XML injection,,N/A,Unknown,Unknown,CWE-91,A05
4,2,2. Web cache poisoning,,N/A,Unknown,Unknown,CWE-436,
5,2,3. Cross-site request forgery,,N/A,Unknown,Unknown,CWE-352,A01
6,2,4. Session token in URL,,N/A,Unknown,Unknown,CWE-200,A01
7,3,2. Password submitted using GET method,,N/A,Unknown,Unknown,CWE-598,
8,3,3. Open redirection (DOM-based),,N/A,Unknown,Unknown,CWE-601,A01
9,3,4. Cookie without HttpOnly flag set,,N/A,Unknown,Unknown,CWE-16,A02


Saved overlap table seed to overlap_table_burp_seed_html_multi_dvwa.csv


,Scan_Run_ID,Target_App,Canonical_Vuln_Key,URL_or_Endpoint,Parameter,Vuln_Type_Label,CWE_ID,OWASP_2025_Category,Detected_by_BurpSuite,Detected_by_OWASP_ZAP,Detected_by_Nikto,Overlap_Pattern
0,DVWA_BURP_01_2026_05_25_BURP_DVWA_Report_4,DVWA,DVWA::N/A:1. Cross-site scripting (reflected),,N/A,1. Cross-site scripting (reflected),CWE-79,A05,Yes,No,No,BurpSuite_only
1,DVWA_BURP_01_2026_05_25_BURP_DVWA_Report_4,DVWA,DVWA::N/A:2. Cross-site scripting (DOM-based),,N/A,2. Cross-site scripting (DOM-based),CWE-79,A05,Yes,No,No,BurpSuite_only
2,DVWA_BURP_01_2026_05_25_BURP_DVWA_Report_4,DVWA,DVWA::N/A:3. Cleartext submission of password,,N/A,3. Cleartext submission of password,CWE-319,A04,Yes,No,No,BurpSuite_only
3,DVWA_BURP_01_2026_05_25_BURP_DVWA_Report_4,DVWA,DVWA::N/A:1. XML injection,,N/A,1. XML injection,CWE-91,A05,Yes,No,No,BurpSuite_only
4,DVWA_BURP_01_2026_05_25_BURP_DVWA_Report_4,DVWA,DVWA::N/A:2. Web cache poisoning,,N/A,2. Web cache poisoning,CWE-436,,Yes,No,No,BurpSuite_only


heree